In [1]:
pip install sentence-transformers scikit-learn pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

train_df = pd.read_csv("dataset/data.csv")
validation_df = pd.read_csv("dataset/validation_data.csv")

print(train_df.head())
print(train_df.shape)
print(validation_df.shape)

   label                                              title  \
0      1  As U.S. budget fight looms, Republicans flip t...   
1      1  U.S. military to accept transgender recruits o...   
2      1  Senior U.S. Republican senator: 'Let Mr. Muell...   
3      1  FBI Russia probe helped by Australian diplomat...   
4      1  Trump wants Postal Service to charge 'much mor...   

                                                text       subject  \
0  WASHINGTON (Reuters) - The head of a conservat...  politicsNews   
1  WASHINGTON (Reuters) - Transgender people will...  politicsNews   
2  WASHINGTON (Reuters) - The special counsel inv...  politicsNews   
3  WASHINGTON (Reuters) - Trump campaign adviser ...  politicsNews   
4  SEATTLE/WASHINGTON (Reuters) - President Donal...  politicsNews   

                 date  
0  December 31, 2017   
1  December 29, 2017   
2  December 31, 2017   
3  December 30, 2017   
4  December 29, 2017   
(39942, 5)
(4956, 5)


In [3]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\(reuters\)", "", text)
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [4]:
train_df["title_clean"] = train_df["title"].apply(clean_text)
train_df["text_clean"] = train_df["text"].apply(clean_text)

validation_df["title_clean"] = validation_df["title"].apply(clean_text)
validation_df["text_clean"] = validation_df["text"].apply(clean_text)

In [5]:
train_df["content"] = (
    train_df["title_clean"] + " " +
    train_df["text_clean"] + " " +
    train_df["subject"].astype(str).str.lower()
)

validation_df["content"] = (
    validation_df["title_clean"] + " " +
    validation_df["text_clean"] + " " +
    validation_df["subject"].astype(str).str.lower()
)

In [6]:
from sklearn.model_selection import train_test_split

X = train_df["content"]
y = train_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

(31953,) (7989,)


In [7]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
X_train_emb = embedder.encode(
    X_train.tolist(),
    show_progress_bar=True
)

X_test_emb = embedder.encode(
    X_test.tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/999 [00:00<?, ?it/s]

Batches:   0%|          | 0/250 [00:00<?, ?it/s]

In [9]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_emb, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [11]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test_emb)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9445487545374891

Classification Report:

              precision    recall  f1-score   support

           0       0.95      0.94      0.94      3989
           1       0.94      0.95      0.94      4000

    accuracy                           0.94      7989
   macro avg       0.94      0.94      0.94      7989
weighted avg       0.94      0.94      0.94      7989


Confusion Matrix:

[[3742  247]
 [ 196 3804]]


In [12]:
X_val_emb = embedder.encode(validation_df["content"].tolist(), show_progress_bar=True)
validation_df["label"] = model.predict(X_val_emb)

validation_df[["label", "title", "text", "subject", "date"]].to_csv(
    "predictions_embeddings.csv",
    index=False
)

Batches:   0%|          | 0/155 [00:00<?, ?it/s]

In [13]:
predictions_df = pd.read_csv("predictions_embeddings.csv")
print(predictions_df.head())
print(predictions_df["label"].value_counts())
print(predictions_df.shape)

   label                                              title  \
0      1  UK's May 'receiving regular updates' on London...   
1      1  UK transport police leading investigation of L...   
2      1  Pacific nations crack down on North Korean shi...   
3      1  Three suspected al Qaeda militants killed in Y...   
4      1  Chinese academics prod Beijing to consider Nor...   

                                                text    subject  \
0  LONDON (Reuters) - British Prime Minister Ther...  worldnews   
1  LONDON (Reuters) - British counter-terrorism p...  worldnews   
2  WELLINGTON (Reuters) - South Pacific island na...  worldnews   
3  ADEN, Yemen (Reuters) - Three suspected al Qae...  worldnews   
4  BEIJING (Reuters) - Chinese academics are publ...  worldnews   

                  date  
0  September 15, 2017   
1  September 15, 2017   
2  September 15, 2017   
3  September 15, 2017   
4  September 15, 2017   
label
0    3220
1    1736
Name: count, dtype: int64
(4956, 5)
